# Activation Checkpointing From Scratch (No `torch.utils.checkpoint`)

This notebook shows how activation checkpointing works on a **simple deep linear model**.

We will:
1. Build a deep model normally (baseline).
2. Build a custom checkpoint function using `torch.autograd.Function`.
3. Run the same training step with and without checkpointing.
4. Compare memory and runtime.

Idea: baseline keeps many intermediate activations for backward; checkpointing keeps fewer and recomputes forward during backward.

In [7]:
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Torch version:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Torch version: 2.10.0+cu128
Device: cuda


## 1) Baseline model

This model is intentionally deep to make activation storage visible.

In [8]:
class DeepLinearNet(nn.Module):
    def __init__(self, d_in=1024, d_hidden=1024, d_out=10, depth=24):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_hidden)
        self.hidden = nn.ModuleList([nn.Linear(d_hidden, d_hidden) for _ in range(depth)])
        self.out_proj = nn.Linear(d_hidden, d_out)

    def forward_baseline(self, x):
        x = F.relu(self.in_proj(x))
        for layer in self.hidden:
            x = F.relu(layer(x))
        return self.out_proj(x)

## 2) Manual checkpoint implementation

We do not use `torch.utils.checkpoint`.

Mechanics:
- Forward under `torch.no_grad()`: do not store intermediate activations.
- Backward: re-run forward with grad enabled, then backpropagate using incoming gradients.

In [9]:
class ManualCheckpoint(torch.autograd.Function):
    @staticmethod
    def forward(ctx, run_fn, *inputs):
        ctx.run_fn = run_fn
        ctx.save_for_backward(*inputs)
        with torch.no_grad():
            outputs = run_fn(*inputs)
        return outputs

    @staticmethod
    def backward(ctx, *grad_outputs):
        detached_inputs = []
        for x in ctx.saved_tensors:
            x_det = x.detach()
            x_det.requires_grad_(x.requires_grad)
            detached_inputs.append(x_det)

        with torch.enable_grad():
            outputs = ctx.run_fn(*detached_inputs)

        if isinstance(outputs, torch.Tensor):
            outputs = (outputs,)

        torch.autograd.backward(outputs, grad_outputs)

        input_grads = tuple(
            x.grad if x.requires_grad else None
            for x in detached_inputs
        )
        return (None, *input_grads)


def checkpoint(run_fn, *inputs):
    if not any(x.requires_grad for x in inputs):
        return run_fn(*inputs)
    return ManualCheckpoint.apply(run_fn, *inputs)

In [10]:
class DeepLinearNetWithCheckpoint(DeepLinearNet):
    def _run_chunk(self, x, start, end):
        for idx in range(start, end):
            x = F.relu(self.hidden[idx](x))
        return x

    def forward(self, x, use_checkpoint=False, chunk_size=4):
        if chunk_size < 1:
            raise ValueError('chunk_size must be >= 1')

        x = F.relu(self.in_proj(x))
        num_hidden = len(self.hidden)

        if use_checkpoint:
            for start in range(0, num_hidden, chunk_size):
                end = min(start + chunk_size, num_hidden)

                def run_fn(t, start=start, end=end):
                    return self._run_chunk(t, start, end)

                x = checkpoint(run_fn, x)
        else:
            x = self._run_chunk(x, 0, num_hidden)

        return self.out_proj(x)


def one_train_step(model, x, y, use_checkpoint=False, chunk_size=4):
    model.zero_grad(set_to_none=True)

    if x.is_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    pred = model(x, use_checkpoint=use_checkpoint, chunk_size=chunk_size)
    loss = F.mse_loss(pred, y)
    loss.backward()
    if x.is_cuda:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    peak_mem_mb = None
    if x.is_cuda:
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return {
        'loss': float(loss.detach()),
        'time_sec': t1 - t0,
        'peak_mem_mb': peak_mem_mb
    }

## 3) Compare baseline vs chunked checkpoint

We clone one initialized model so both paths have identical weights.

We now checkpoint hidden layers in chunks (for example 4 layers per checkpoint call).

Expected:
- Similar loss values.
- Checkpoint path often uses less memory on CUDA for larger models.
- Checkpoint path is slower (extra recomputation).

In [ ]:
torch.manual_seed(0)

batch_size = 32
d_in = 1024
d_hidden = 15536
d_out = 10
depth = 1
chunk_size = 4

x = torch.randn(batch_size, d_in, device=device, requires_grad=True)
y = torch.randn(batch_size, d_out, device=device)

model_base = DeepLinearNetWithCheckpoint(d_in, d_hidden, d_out, depth).to(device)
model_ckpt = copy.deepcopy(model_base).to(device)

base_stats = one_train_step(model_base, x, y, use_checkpoint=False, chunk_size=chunk_size)

# Recreate input so both runs start from same gradient state.
x2 = x.detach().clone().requires_grad_(True)
ckpt_stats = one_train_step(model_ckpt, x2, y, use_checkpoint=True, chunk_size=chunk_size)

print('Chunk size:', chunk_size)
print('Baseline stats:', base_stats)
print('Checkpoint stats:', ckpt_stats)

if base_stats['peak_mem_mb'] is not None and ckpt_stats['peak_mem_mb'] is not None:
    saved = base_stats['peak_mem_mb'] - ckpt_stats['peak_mem_mb']
    print(f'Memory saved: {saved:.2f} MB')

slowdown = ckpt_stats['time_sec'] / max(base_stats['time_sec'], 1e-12)
print(f'Slowdown factor: {slowdown:.2f}x')

Chunk size: 4
Baseline stats: {'loss': 0.9965780377388, 'time_sec': 0.2356083239428699, 'peak_mem_mb': 3222.08056640625}
Checkpoint stats: {'loss': 0.9965780377388, 'time_sec': 0.35485413670539856, 'peak_mem_mb': 4205.4736328125}
Memory saved: -983.39 MB
Slowdown factor: 1.51x


: 

## 4) What happened under the hood

For each checkpointed hidden layer:
- In forward, we only save the layer input tensor.
- In backward, PyTorch calls our custom `backward`, which recomputes that layer forward.
- Gradients are computed from the recomputed outputs.

This trades **memory** for **compute**:
- Lower activation memory peak.
- Extra forward compute during backward.

You can increase `depth` and `batch_size` to amplify the difference.